# Sentinel-2 L2A imagery — TESSERA tile grid, 2023

**Purpose.** Download the raw Sentinel-2 scenes for the conventional
(non-foundation-model) baseline in RQ1 (see `docs/research_questions.md`),
resampled onto the exact same 10 m grid as a TESSERA embeddings tile,
so Sentinel-2 and the embeddings line up pixel for pixel.

**Area.** One TESSERA tile (`geotessera download --bbox
"-8.46,40.21,-8.44,40.29" --year 2023`, see
`data/raw/satellite/tessera_tiles/coimbra_2023_geotiff/`), read only
for its grid (CRS, transform, shape) — a narrow strip, 1.75 x 8.89 km,
889 x 175 px at 10 m, EPSG:32629. This is a quick pipeline check on a
small area, not the full pilot extent in `data/README.md` — the tile
and/or area can be widened once this works.

**Data.** Sentinel-2 L2A (surface reflectance, atmospherically
corrected) Cloud-Optimized GeoTIFFs from the AWS Earth Search STAC
catalog (`https://earth-search.aws.element84.com/v1`,
`sentinel-2-l2a` collection), no account needed.

- **Tile/period/filter:** MGRS tile `29TNE`, 2023-01-01 to 2023-12-31,
  `eo:cloud_cover < 20%` — 61 scenes.
- **Bands:** blue, green, red, nir, rededge1–3, nir08 (B8A), swir16,
  swir22, scl.
- **Access method:** `odc.stac.load`, one call per scene, `geobox=`
  the TESSERA tile's own geobox (`tess.odc.geobox`) instead of
  `bbox=`/`crs=`/`resolution=` — this reads only the pixels inside
  that geobox directly from the remote COGs (HTTP range reads via
  GDAL/rasterio), no full-tile download.
- **Resampling onto the TESSERA grid:** replaces the earlier plan to
  keep each band at its native resolution — every band (10 m and
  20 m) is now resampled onto the same 10 m TESSERA grid, needed to
  compare/combine Sentinel-2 features and TESSERA embeddings per
  pixel. Bilinear for the continuous reflectance bands, nearest for
  `scl` (categorical).
- **No precedent found in the literature read so far for this exact
  access route**; Sentinel-2 as the baseline sensor is grounded in
  Mouta et al. (2021) and Domingo et al. (2023) (see
  `docs/methodology.md`).

**Only downloads what is missing** — each band file is skipped if it
already exists, and the whole search step is skipped if
`manifest.csv` already exists.

**Output:** one GeoTIFF per band per scene, on the TESSERA grid, in
`data/raw/satellite/sentinel2/tessera_grid_2023/<scene_id>/<band>.tif`,
plus `manifest.csv` (scene id, datetime, cloud cover) in the same
folder. Resampled onto a common grid, so — unlike the other raw
satellite data in this project — this is already a light processing
step, not a byte-for-byte original; kept under `raw/` here because it
is still per-scene, unmosaicked, unmasked reflectance, but flag this
if it is later felt to belong in `interim/` instead.

**Access date:** 2026-11-03.


In [ ]:
import time
from pathlib import Path

import odc.geo.xr  # noqa: F401 (registers the .odc accessor)
import odc.stac
import pandas as pd
import rioxarray
from pystac_client import Client

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
TESS_TIF = ROOT / "data/raw/satellite/tessera_tiles/coimbra_2023_geotiff/tessera_2023_utm29.tif"
OUT_DIR = ROOT / "data" / "raw" / "satellite" / "sentinel2" / "tessera_grid_2023"
MANIFEST_FILE = OUT_DIR / "manifest.csv"

STAC_URL = "https://earth-search.aws.element84.com/v1"
TILE = "MGRS-29TNE"
CLOUD_COVER_LT = 20
DATE_RANGE = "2023-01-01/2023-12-31"
CONTINUOUS_BANDS = ["blue", "green", "red", "nir", "rededge1", "rededge2", "rededge3",
                    "nir08", "swir16", "swir22"]
CATEGORICAL_BANDS = ["scl"]

tess = rioxarray.open_rasterio(TESS_TIF)
GEOBOX = tess.odc.geobox
print("geobox:", GEOBOX)


In [ ]:
def search_items():
    catalog = Client.open(STAC_URL)
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        query={"grid:code": {"eq": TILE}, "eo:cloud_cover": {"lt": CLOUD_COVER_LT}},
        datetime=DATE_RANGE,
    )
    return list(search.items())


def write_band_group(item, bands, resampling, scene_dir, retries=3):
    missing = [b for b in bands if not (scene_dir / f"{b}.tif").exists()]
    if not missing:
        return
    for attempt in range(retries):
        try:
            ds = odc.stac.load([item], bands=missing, geobox=GEOBOX, resampling=resampling, chunks={})
            ds = ds.isel(time=0)
            break
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(5)
    scene_dir.mkdir(parents=True, exist_ok=True)
    for band in missing:
        da = ds[band].rio.write_crs(GEOBOX.crs)
        da.rio.to_raster(scene_dir / f"{band}.tif", compress="deflate")


def download_scene(item):
    scene_dir = OUT_DIR / item.id
    write_band_group(item, CONTINUOUS_BANDS, "bilinear", scene_dir)
    write_band_group(item, CATEGORICAL_BANDS, "nearest", scene_dir)


def download_scenes(items):
    rows = []
    for item in items:
        t0 = time.time()
        download_scene(item)
        rows.append(dict(scene_id=item.id, datetime=item.properties["datetime"],
                          cloud_cover=item.properties["eo:cloud_cover"]))
        print(item.id, f"{time.time() - t0:.1f}s")
    return pd.DataFrame(rows)


In [ ]:
# Sanity check: one scene should take seconds, not minutes.
items = search_items()
print(f"{len(items)} scenes found")

t0 = time.time()
download_scene(items[0])
print(f"first scene: {time.time() - t0:.1f}s")


In [ ]:
if not MANIFEST_FILE.exists():
    manifest = download_scenes(items)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest.to_csv(MANIFEST_FILE, index=False)
else:
    manifest = pd.read_csv(MANIFEST_FILE)

print(f"\n{len(manifest)} scenes, tile {TILE}, {DATE_RANGE}, cloud < {CLOUD_COVER_LT}%")


In [ ]:
import rasterio

sample_scene = OUT_DIR / manifest.iloc[0]["scene_id"]
with rasterio.open(sample_scene / "red.tif") as src:
    print("red shape:", src.shape, "crs:", src.crs, "res:", src.res)
with rasterio.open(sample_scene / "scl.tif") as src:
    print("scl shape:", src.shape, "crs:", src.crs, "res:", src.res)
